# Assignment 10 — Audio Classification with STT and TTS

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** GPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Application

Build a small voice-command assistant. Audio is converted to a
spectrogram and classified by a CNN. The predicted command is the
limited-vocabulary **Speech-to-Text (STT)** output. Google Text-to-Speech
then speaks a response.

Dataset: TensorFlow's Mini Speech Commands (eight one-second words).
Evaluation uses accuracy, macro F1, confusion matrix, and word error rate
(WER). Here each utterance contains one word, so WER equals the fraction
of incorrectly transcribed commands.


In [ ]:
%pip install -q -U gTTS
import pathlib
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from IPython.display import Audio, display
from gtts import gTTS
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

url = "http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip"
archive = tf.keras.utils.get_file("mini_speech_commands.zip", origin=url, extract=True,
                                  cache_dir=".", cache_subdir="data")
data_dir = pathlib.Path(archive).parent / "mini_speech_commands"
commands = np.array(sorted([p.name for p in data_dir.iterdir() if p.is_dir()]))
print("Commands:", commands)


In [ ]:
train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=data_dir, batch_size=64, validation_split=0.20,
    seed=SEED, output_sequence_length=16000, subset="both"
)
# Split validation batches into validation and final test sets.
val_batches = tf.data.experimental.cardinality(val_ds)
test_ds = val_ds.take(val_batches // 2)
val_ds = val_ds.skip(val_batches // 2)

def squeeze(audio, labels):
    return tf.squeeze(audio, axis=-1), labels
train_ds = train_ds.map(squeeze, tf.data.AUTOTUNE)
val_ds = val_ds.map(squeeze, tf.data.AUTOTUNE)
test_ds = test_ds.map(squeeze, tf.data.AUTOTUNE)


In [ ]:
def waveform_to_spectrogram(waveform):
    stft = tf.signal.stft(waveform, frame_length=255, frame_step=128)
    magnitude = tf.abs(stft)
    return magnitude[..., tf.newaxis]

def to_spectrogram(audio, label):
    return waveform_to_spectrogram(audio), label

spec_train = train_ds.map(to_spectrogram, tf.data.AUTOTUNE).cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
spec_val = val_ds.map(to_spectrogram, tf.data.AUTOTUNE).cache().prefetch(tf.data.AUTOTUNE)
spec_test = test_ds.map(to_spectrogram, tf.data.AUTOTUNE).cache().prefetch(tf.data.AUTOTUNE)

sample_specs, sample_labels = next(iter(spec_train))
input_shape = sample_specs.shape[1:]
plt.imshow(tf.math.log(sample_specs[0, ..., 0] + 1e-6).numpy().T,
           origin="lower", aspect="auto", cmap="magma")
plt.title(f"Spectrogram: {commands[int(sample_labels[0])]}" )
plt.xlabel("Time frame"); plt.ylabel("Frequency bin"); plt.show()


In [ ]:
normalization = tf.keras.layers.Normalization()
normalization.adapt(spec_train.map(lambda x, y: x))

model = tf.keras.Sequential([
    tf.keras.layers.Input(input_shape),
    tf.keras.layers.Resizing(64, 64),
    normalization,
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.35),
    tf.keras.layers.Dense(len(commands), activation="softmax"),
], name="speech_command_cnn")
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
history = model.fit(
    spec_train, validation_data=spec_val, epochs=15,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )],
)


In [ ]:
probabilities = model.predict(spec_test, verbose=0)
y_pred = probabilities.argmax(axis=1)
y_true = np.concatenate([labels.numpy() for _, labels in spec_test])

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
word_error_rate = np.mean(y_true != y_pred)  # one reference word per recording
print(f"Accuracy: {accuracy:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"One-word WER: {word_error_rate:.3f}")
print(classification_report(y_true, y_pred, target_names=commands, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=commands, yticklabels=commands)
plt.xlabel("Predicted/STT word"); plt.ylabel("Actual word")
plt.title("Voice-command confusion matrix"); plt.show()


In [ ]:
# Demonstrate the complete audio -> STT -> response -> TTS pipeline.
waveforms, actual_labels = next(iter(test_ds))
one_waveform = waveforms[0]
one_spec = waveform_to_spectrogram(one_waveform)[tf.newaxis, ...]
predicted_id = int(model.predict(one_spec, verbose=0).argmax(axis=1)[0])
recognized_text = commands[predicted_id]

responses = {
    "yes": "You said yes.", "no": "You said no.",
    "up": "Moving up.", "down": "Moving down.",
    "left": "Turning left.", "right": "Turning right.",
    "go": "Starting now.", "stop": "Stopping now.",
}
response_text = responses.get(recognized_text, f"I heard {recognized_text}.")
print("Actual:", commands[int(actual_labels[0])])
print("STT output:", recognized_text)
print("Assistant response:", response_text)
display(Audio(one_waveform.numpy(), rate=16000))

tts_path = "assistant_response.mp3"
gTTS(response_text, lang="en").save(tts_path)
display(Audio(tts_path, autoplay=False))


## Discussion and limitation

The CNN recognizes only eight known words, so this is command-level STT,
not unrestricted transcription. A full assistant could replace it with
Whisper or another speech-recognition model. TTS quality can be discussed
using intelligibility, naturalness (a listener rating), and response time;
classification quality is captured by accuracy, macro F1, and WER.


## Conclusion

The experiment above provides a complete training and evaluation workflow. The
printed metrics and plots are the result for the current run and should be used
to identify the strongest behaviour, the main limitation, and one justified
improvement. Exact values may vary slightly because neural-network training is
stochastic.
